In [1]:
from pathlib import Path

import numpy as np
import time
import sys
import json
from tqdm.notebook import tqdm
import itertools

from SolverV9 import (
    LiouvilleSpectroscopySolver,
    SpectroscopyPlotter,
    standard_nq_protocol,
)

from IPython.display import display, HTML
display(HTML("<style>div.output_area pre, div.jp-OutputArea-output pre {white-space: pre;}</style>"))

np.set_printoptions(threshold=sys.maxsize, linewidth=np.inf, precision=5, suppress=True)

### Coupled Model

### Solver Parameters

- `T`: System temperature. `T = 0` assumes that the system initially occupies its ground state.
- `Eta`: Homogeneous broadening used in the resolvents, in eV. It primarily controls the spectral linewidth.
- `backend`: Numerical implementation used by the solver. `"dense"` uses dense matrices and NumPy linear-algebra routines.

In [2]:
solver_params = {
    "T": 0.0,
    "Eta": 0.000001,
    "backend": "dense",
    "parallel_backend": "threading",
    "n_jobs": -1,
}

In [3]:
hbar = 0.658211951  # in eV fs

en_b = 1.4907 # bright state excitonic resonant energy in eV
en_d = en_b -0.001 # dark state excitonic resonant energy in eV
detuning = -0.002 # detuning between exciton and cavity in eV
en_cav = en_b + detuning # cavity resonant energy in eV

rabi = 0.0004  # vaccum rabi splitting
g = rabi / 2  # coupling strength for light-matter interaction in eV
J = 0.0002 # coupling strength between XX and XXb
n_th = 0. # average number of cavity photons

mu_qm1 = 0.3 # dipole strength for C
mu_qm2 = 0.29  # dipole strength for CC
mu_b1 = 0.04  # dipole strength for B
mu_b2 = 0.25  # dipole strength for BB

kappa = 0.0011  # cavity decay strength
gamma_b_decay = 0.00009  # heavy exciton decay strength
gamma_d_decay = 0.000015  # light exciton decay strength
gamma_cav_phase = 0.00001  # cavity dephasing strength
gamma_exc_phase = 0.00001  # exciton dephasing strength
gamma_bd = 0.0001 # b-d swapping strength
gamma_bc = 0.0001 # b-c swapping strength

bb_bind=0.00086  # strength of BB biexciton binding
dd_bind=0.#00086  # strength of DD biexciton binding

grid = 100
t2 = 50

date = time.strftime('%Y-%m-%d')
hour = time.strftime("%Hh%Mm%S")

### 0-quantum + 1-quantum + 2-quantum
loop over i for any parameter

In [4]:
param = "mu_b1"
rang = ""
rang = [0.001,0.04,10]
# kappa_list = [0.0011, 0.0015, 0.0019]
for i in tqdm(np.linspace(*rang)):
# for i in tqdm(np.geomspace(*rang)):
# for i in tqdm(list(itertools.product(np.linspace(*rang), np.linspace(*rang)))):
# for i in tqdm(list(itertools.product(np.linspace(*rang), np.linspace(*rang)))):
# for i in tqdm(list(itertools.product(np.linspace(*rang), np.linspace(*rang), np.linspace(*rang)))):
# for i in tqdm(list(itertools.product(np.linspace(*rang), np.linspace(*rang), np.linspace(rang[0], rang[1], 6), np.linspace(*rang)))):
# for i in [1.]:
#     mu_qm1 = i[0]
#     mu_qm2 = i[1]
#     mu_d1 = i[2]
#     mu_d2 = i[3]
    mu_b1 = i
    # gamma_b_decay = i[1]
    # gamma_cav_phase = i[2]
    # en_cav=en_b + detuning

    # setting up dm
    exc_basis = np.zeros((8, 8))
    exc_basis[0, 0] = 1 # ground,B,C,D,BB,BC,CC,DD
    rho = exc_basis @ exc_basis.T  # ground state of Hamiltonian

    wc = en_cav / hbar
    wb = en_b / hbar  # bright exciton resonant frequencies
    wd = en_d / hbar  # dark exciton resonant frequencies

    # each individual lowering operators // if 2 different species, ex. bc, the 2nd letter is the one that is lowering: bc_low gives |bc> to |b>
    b_low = np.zeros_like(exc_basis)
    b_low[0, 1] = 1

    c_low = np.zeros_like(exc_basis)
    c_low[0, 2] = 1

    d_low = np.zeros_like(exc_basis)
    d_low[0, 3] = 1

    bb_low = np.zeros_like(exc_basis)
    bb_low[1, 4] = 1

    bc_low = np.zeros_like(exc_basis)
    bc_low[1, 5] = 1 / np.sqrt(2)

    cb_low = np.zeros_like(exc_basis)
    cb_low[2, 5] = 1 / np.sqrt(2)

    cc_low = np.zeros_like(exc_basis)
    cc_low[2, 6] = 1

    dd_low = np.zeros_like(exc_basis)
    dd_low[3, 7] = 1

    # global lowering operators
    cav_low = c_low + bc_low + cc_low
    exc_b_low = b_low + bb_low + cb_low
    exc_d_low = d_low + dd_low

    H0_b = b_low.T @ b_low  # b-pop
    H0_c = c_low.T @ c_low  # c-pop
    H0_d = d_low.T @ d_low  # d-pop
    H0_bb = bb_low.T @ bb_low  # bb-pop
    H0_bc = bc_low.T @ bc_low + cb_low.T @ cb_low  # bc-pop
    H0_cc = cc_low.T @ cc_low  # cc-pop
    H0_dd = dd_low.T @ dd_low  # dd-pop

    H0 = (en_b * H0_b + en_cav * H0_c + en_d * H0_d + (2 * en_b - bb_bind) * H0_bb +
          (en_b + en_cav) * H0_bc + 2 * en_cav * H0_cc + (2 * en_d - dd_bind) * H0_dd)

    H_coupling = b_low.T @ d_low  # 1Q B-D coupling
    H_coupling += np.sqrt(2) * (bb_low.T @ b_low.T @ d_low @ dd_low)  # 2Q B-D coupling
    H_coupling += H_coupling.T

    H_int = b_low.T @ c_low  # 1Q light-matter coupling
    H_int += 2 * (bb_low.T @ bc_low + cb_low.T @ cc_low) # 2Q bright light-matter coupling
    H_int += H_int.T

    H = H0 + g * H_int + J * H_coupling  # total hamiltonian
    # print("H0", H0)
    # print("H_int\n", H_int)
    # print("H_coupling\n", H_coupling)
    # print("H", H)


    ad = mu_qm2 * cav_low + (mu_qm1 - mu_qm2) * c_low + mu_b2 * exc_b_low + (mu_b1 - mu_b2) * b_low # lowering operator
    # ad = mu_cav * cav_low + mu_exc * ((1 - n_dens) * (exc_h_low + exc_l_low) + n_dens * (h_low + ch_low + l_low + cl_low)) # lowering operator
    obs = ad + ad.T
    mu = obs

    if isinstance(i, list) and i == rang[0] or i == 1.:
        print("mud\n", mu)


    # collapse operators
    c_b_rel = np.sqrt(gamma_b_decay * (n_th + 1)) * b_low # exciton relaxation
    c_b_exc = np.sqrt(gamma_b_decay * n_th) * b_low.T # exciton excitation
    c_c_rel = np.sqrt(kappa * (n_th + 1)) * c_low # cavity relaxation
    c_c_exc = np.sqrt(kappa * n_th) * c_low.T # cavity excitation
    c_d_rel = np.sqrt(gamma_d_decay * (n_th + 1)) * d_low # exciton relaxation
    c_d_exc = np.sqrt(gamma_d_decay * n_th) * d_low.T # exciton excitation
    c_bb_rel = np.sqrt(gamma_b_decay * (n_th + 1)) * bb_low # exciton relaxation
    c_bb_exc = np.sqrt(gamma_b_decay * n_th) * bb_low.T # exciton excitation
    c_cb_rel = np.sqrt(gamma_b_decay * (n_th + 1)) * cb_low # cavity relaxation
    c_cb_exc = np.sqrt(gamma_b_decay * n_th) * cb_low.T # cavity excitation
    c_bc_rel = np.sqrt(kappa * (n_th + 1)) * bc_low # exciton relaxation
    c_bc_exc = np.sqrt(kappa * n_th) * bc_low.T # exciton excitation
    c_cc_rel = np.sqrt(kappa * (n_th + 1)) * cc_low # cavity relaxation
    c_cc_exc = np.sqrt(kappa * n_th) * cc_low.T # cavity excitation
    c_dd_rel = np.sqrt(gamma_d_decay * (n_th + 1)) * dd_low # exciton relaxation
    c_dd_exc = np.sqrt(gamma_d_decay * n_th) * dd_low.T # exciton excitation

    c_b_dep = np.sqrt(gamma_exc_phase) * en_b * H0_b # B pop
    c_d_dep = np.sqrt(gamma_exc_phase) * en_d * H0_d # D pop
    c_c_dep = np.sqrt(gamma_cav_phase) * en_cav * H0_c # C pop
    c_bb_dep = np.sqrt(gamma_exc_phase) * (2 * en_b - bb_bind) * H0_bb # BB pop
    c_bc_dep = (np.sqrt(gamma_cav_phase) + np.sqrt(gamma_exc_phase)) * (en_b + en_cav) / 2 * H0_bc # BC pop
    c_cc_dep = np.sqrt(gamma_cav_phase) * 2 * en_cav * H0_cc # CC pop
    c_dd_dep = np.sqrt(gamma_exc_phase) * (2 * en_d - dd_bind ) * H0_dd # DD pop

    c_b_d = np.sqrt(gamma_bd) * exc_d_low.T @ exc_b_low # bright exciton to dark exciton transition
    c_d_b = np.sqrt(gamma_bd) * exc_b_low.T @ exc_d_low # dark exciton to bright exciton transition

    c_b_cav = np.sqrt(gamma_bc) * cav_low.T @ exc_b_low # bright exciton to cavity photon transition
    c_cav_b = np.sqrt(gamma_bc) * exc_b_low.T @ cav_low # cavity photon to bright exciton transition

    # c_ops = [(c_b_rel,1), (c_b_exc,1), (c_c_rel,1), (c_c_exc,1), (c_d_rel,1), (c_d_exc,1), (c_bb_rel,1), (c_bb_exc,1), (c_cb_rel,1), (c_cb_exc,1), (c_bc_rel,1), (c_bc_exc,1), (c_cc_rel,1), (c_cc_exc,1), (c_dd_rel,1), (c_dd_exc,1), (c_b_dep,1), (c_c_dep,1), (c_d_dep,1), (c_bc_dep,1), (c_cc_dep,1), (c_dd_dep,1)]
    c_ops = [(c_b_rel,1), (c_c_rel,1), (c_d_rel,1), (c_bb_rel,1), (c_cb_rel,1), (c_bc_rel,1), (c_cc_rel,1), (c_dd_rel,1), (c_b_dep,1), (c_c_dep,1), (c_d_dep,1), (c_bb_dep, 1), (c_bc_dep,1), (c_cc_dep,1), (c_dd_dep,1)]

    # solver
    solver = LiouvilleSpectroscopySolver(solver_params)
    solver.feed_model(
        H,
        mu,
        c_ops_raw=c_ops,
        initial_density_matrix=rho,
        density_matrix_basis="site",
    )

    # pathways
    pathways = solver.generate_pathways_with_ufss(
        "-++",
        maximum_manifold=2,
        component="chi3_1q",
    )

    protocol = standard_nq_protocol(
        order=1,
        nq_interval=1,
        detection_interval=3,
        n_interactions=3,
        nq_axis="omega_1q",
        detection_axis="omega_emit",
    )

    [(pathway.name, pathway.interactions, pathway.coherence_orders) for pathway in pathways]

    # generate the spectrum
    omega_1q = np.linspace(-1.495, -1.482, grid)
    omega_emit = np.linspace(1.482, 1.495, grid)

    delays = {
        "t2": t2 / hbar
    }

    result = solver.generate_NQ_spectrum(
        1,
        protocol,
        axes={"omega_1q": omega_1q, "omega_emit": omega_emit},
        delays=delays,
        pathways=pathways
        )

    save_pdf = True
    output_directory = Path.cwd() / "Result_Test" / "Multiorder_pathway_plot__bd" / date / f"{hour}_{param}{rang}"
    output_directory.mkdir(parents=True, exist_ok=True)

    if isinstance(i, (int, float)):
        spec_real = f"chi3_Q1_real_{param}{i:.6f}.png"
        spec_imag = f"chi3_Q1_imag_{param}{i:.6f}.png"
        spec_abs = f"chi3_Q1_abs_{param}{i:.6f}.png"
    else:
        spec_real = f"chi3_Q1_real_{param}" + "_".join(f"{x:.6f}" for x in i) + ".png"
        spec_imag = f"chi3_Q1_imag_{param}" + "_".join(f"{x:.6f}" for x in i) + ".png"
        spec_abs = f"chi3_Q1_abs_{param}" + "_".join(f"{x:.6f}" for x in i) + ".png"

    plotter = SpectroscopyPlotter(detection_phase=np.pi/2)
    plot_result = plotter.plot_pathways_multiorder(
        result,
        pathways="all",
        totals=["1Q"],
        view="real",
        normalization="none",
        axis_labels={
            "omega_1q": "1Q energy (eV)",
            "omega_emit": "Emission energy (eV)",
        },
        include_diagrams=False,
        display_diagrams=False,
        save_pdf=save_pdf,
        output_directory=output_directory if save_pdf else None,
        spectrum_pdf_name=spec_real,
        show=False,
    )

    plot_result = plotter.plot_pathways_multiorder(
        result,
        pathways="all",
        totals=["1Q"],
        view="imag",
        normalization="none",
        axis_labels={
            "omega_1q": "1Q energy (eV)",
            "omega_emit": "Emission energy (eV)",
        },
        include_diagrams=False,
        display_diagrams=False,
        save_pdf=save_pdf,
        output_directory=output_directory if save_pdf else None,
        spectrum_pdf_name=spec_imag,
        show=False,
    )

    plot_result = plotter.plot_pathways_multiorder(
        result,
        pathways="all",
        totals=["1Q"],
        view="abs",
        normalization="none",
        axis_labels={
            "omega_1q": "1Q energy (eV)",
            "omega_emit": "Emission energy (eV)",
        },
        include_diagrams=False,
        display_diagrams=False,
        save_pdf=save_pdf,
        output_directory=output_directory if save_pdf else None,
        spectrum_pdf_name=spec_abs,
        show=False,
    )

    # plotter.plot_spectrum_result_contours(
    #     result,
    #     spectra="pathways",
    #     names=["R1", "R2", "R3"],
    #     totals="selected",
    #     normalization="panel",
    #     color_map="PuOr",
    #     levels=25,
    #     show=False,
    # )
    print(f"{i} diagram done\n")

  0%|          | 0/10 [00:00<?, ?it/s]

--- Model loading ---
Model transformed to the eigenbasis.
Liouville backend ready: dense.
Calculating 3 pathway spectrum/s on a 100x100 grid with protocol 'standard_1q' using parallel=threading, n_jobs=-1.
Using dense prefix-tree pathway reuse.
0.001 diagram done

--- Model loading ---
Model transformed to the eigenbasis.
Liouville backend ready: dense.
Calculating 3 pathway spectrum/s on a 100x100 grid with protocol 'standard_1q' using parallel=threading, n_jobs=-1.
Using dense prefix-tree pathway reuse.
0.005333333333333333 diagram done

--- Model loading ---
Model transformed to the eigenbasis.
Liouville backend ready: dense.
Calculating 3 pathway spectrum/s on a 100x100 grid with protocol 'standard_1q' using parallel=threading, n_jobs=-1.
Using dense prefix-tree pathway reuse.
0.009666666666666667 diagram done

--- Model loading ---
Model transformed to the eigenbasis.
Liouville backend ready: dense.
Calculating 3 pathway spectrum/s on a 100x100 grid with protocol 'standard_1q' us

In [5]:
with open(f"{output_directory}/parameters.txt", "w", encoding="utf-8") as file:
    file.write(hour+"\n")
    file.write(f"Energies  bright={en_b:.5f}, dark={en_d:.5f}, cavity={en_cav:.5f}\n")
    file.write(f"detuning delta={detuning:.4}\n")
    file.write(f"light-matter coupling g={g:.4f}\n")
    file.write(f"bright-dark coupling J={J:.4f}\n")
    file.write(f"dephasing  excitons={gamma_exc_phase:.4f}, cavity={gamma_cav_phase:.4f}\n")
    file.write(f"decay  bright={gamma_b_decay:.4f}, dark={gamma_d_decay:.4f}, cavity={kappa:.4f}\n")
    file.write(f"binding energies  bright-bright={bb_bind:.4f}, dark-dark={dd_bind:.4f}\n")
    file.write(f"swapping strength  bright-dark={gamma_bd:.4f}, bright-cavity={gamma_bc:.4f}\n")
    file.write(f"broadening eta={solver_params["Eta"]}\n")
    file.write(f"grid size(f1,f3)={grid}\n")
    file.write(f"population time={t2}\n")
    file.write(f"average photons={n_th}\n")
    for ii in [H, H_int, H_coupling, mu]+[cc[0] for cc in c_ops]:
        for name, val in list(locals().items()):
            if val is ii:
                file.write(f"\n\n{name}\n")
                break
        for i, row in enumerate(ii.tolist()):
            formatted = [f"{v.real:.5f}" if v.real >= 0.000001 else "0" for v in row]
            file.write("[" + ", ".join(f"{s:>{8}}" for s in formatted) + "]" + "\n")

In [6]:
# print("Plotted panels:", plot_result.panel_names)
# print("Matching UFSS diagrams:", tuple(plot_result.diagrams))
# if save_pdf:
#     print("Spectrum PDF:", plot_result.spectrum_pdf)
#     print("Diagram PDFs:", plot_result.diagram_paths)
